# Project 3

## 1. Imports and Spark session

Reused from Project 2


In [1]:
import os
import sys
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType
)

spark = (
    SparkSession.builder
    .appName("project3")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


Seeding PostgreSQL source database


OperationalError: connection to server at "postgres" (192.168.167.4), port 5432 failed: FATAL:  password authentication failed for user "cdc_user"


## 2. Configuration

The naming below separates the two Project 3 paths:

- **CDC path**: PostgreSQL → Debezium → Kafka → Bronze CDC → Silver CDC
- **Taxi path**: Kafka → Bronze taxi → Silver taxi → Gold taxi


In [ ]:
KAFKA_BOOTSTRAP = "kafka:9092"

CDC_TOPIC_PATTERN = r"dbserver1\.public\.(customers|drivers)"
TAXI_TOPIC = "taxi-trips"

DB = "lakehouse.project3"

BRONZE_CDC_TABLE = f"{DB}.bronze_cdc"
SILVER_CUSTOMERS_TABLE = f"{DB}.silver_customers"
SILVER_DRIVERS_TABLE = f"{DB}.silver_drivers"

BRONZE_TAXI_TABLE = f"{DB}.bronze_taxi"
SILVER_TAXI_TABLE = f"{DB}.silver_taxi"
GOLD_TAXI_TABLE = f"{DB}.gold_taxi_hourly_zone"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")


## 3. Lookup data from Project 2

In [ ]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")

pickup_zones = (
    zones
    .select(
        F.col("LocationID").alias("pu_location_id"),
        F.col("Zone").alias("pickup_zone"),
        F.col("Borough").alias("pickup_borough")
    )
)

dropoff_zones = (
    zones
    .select(
        F.col("LocationID").alias("do_location_id"),
        F.col("Zone").alias("dropoff_zone"),
        F.col("Borough").alias("dropoff_borough")
    )
)

zones.show(5, truncate=False)


## 4. Taxi trip schema from Project 2


In [ ]:
trip_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
])


## 5. Table definitions

Schema shell is created here.  


In [ ]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BRONZE_CDC_TABLE} (
    source_table STRING,
    record_id INT,
    op STRING,
    before_json STRING,
    after_json STRING,
    source_lsn BIGINT,
    ts_ms BIGINT,
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    is_tombstone BOOLEAN,
    bronze_ingested_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (source_table)
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SILVER_CUSTOMERS_TABLE} (
    id INT,
    name STRING,
    email STRING,
    country STRING,
    created_at TIMESTAMP
)
USING iceberg
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SILVER_DRIVERS_TABLE} (
    id INT,
    name STRING,
    license_number STRING,
    rating DOUBLE,
    city STRING,
    active BOOLEAN,
    created_at TIMESTAMP
)
USING iceberg
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BRONZE_TAXI_TABLE} (
    kafka_key STRING,
    raw_json STRING,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    bronze_ingested_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(kafka_timestamp))
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SILVER_TAXI_TABLE} (
    trip_id STRING,
    vendor_id INT,
    pickup_ts TIMESTAMP,
    dropoff_ts TIMESTAMP,
    pickup_date DATE,
    pickup_hour INT,
    trip_duration_minutes DOUBLE,
    passenger_count INT,
    trip_distance DOUBLE,
    pu_location_id INT,
    pickup_zone STRING,
    pickup_borough STRING,
    do_location_id INT,
    dropoff_zone STRING,
    dropoff_borough STRING,
    fare_amount DOUBLE,
    tip_amount DOUBLE,
    total_amount DOUBLE,
    payment_type INT,
    congestion_surcharge DOUBLE,
    raw_json STRING
)
USING iceberg
PARTITIONED BY (pickup_date)
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_TAXI_TABLE} (
    pickup_date DATE,
    pickup_hour INT,
    pickup_zone STRING,
    trip_count BIGINT,
    avg_fare_amount DOUBLE,
    avg_total_amount DOUBLE,
    avg_trip_distance DOUBLE,
    avg_trip_duration_minutes DOUBLE
)
USING iceberg
PARTITIONED BY (pickup_date)
''')
